# NB6 — Tables

v7 pipeline. Runs **locally**, no GPU or internet needed (pandas / numpy / scipy only).

## Expected local folder layout

```
V2/
  notebook/
    NB6_table.ipynb        <- this notebook
  kaggle-output/
    NB1-outputs/            (from NB1, downloaded from Kaggle)
    NB2-outputs/
      NB2-result/           (from NB2)
      NB2-models/
    NB3-outputs/
      NB3-result/           (from NB3)
      NB3-models/
    NB4-outputs/            (from NB4)
    NB5-output/             (from NB5)
  result/
    table/                  <- this notebook writes here
    figure/
```

## What this notebook builds

All statistical tables for the final report, from the raw per-sample artifacts produced on
Kaggle. Nothing here needs to be re-run on Kaggle — everything is a pure re-analysis of already
computed predictions, uncertainty estimates, and SHAP/LIME attributions.

| Table | Content | Answers |
|---|---|---|
| table1_model_performance | AUC/AUPRC/ECE/NLL/Brier, balanced + natural test | overview |
| table2_faithfulness | Comp/Suff, 3 variants x SHAP/LIME x K grid | RQ4 (metric consistency) |
| table2b_stability | reproducibility Jaccard + local-robustness | explainer quality |
| table3_spearman_uncertainty_components | raw Spearman, all 4 uncertainty components, all/minority/majority | RQ1, RQ2 |
| table3b_fisher_z_minority_majority | corrected Fisher Z (Spearman SE, x1.06) | RQ2 |
| table4_gating_metrics | Gating AUC (standard + class-discriminative), AURC | RQ1, RQ3 |
| table5_budget_grid_summary | M x T sweep with CI | RQ3 |
| table6_uq_distribution | descriptive stats of the 4 uncertainty components | UQ overview |
| table7_partial_correlation | partial Spearman controlling p(1-p), vs aleatoric, 3 variants | RQ1, RQ4 |
| table8_spearman_sensitivity_K | Spearman across the full K grid, not just the median K | robustness check |
| table9_usfg_bootstrap | USFG with bootstrap CI + Mann-Whitney, 4 components (raw `prob` Comp, historical reference) | RQ1, RQ2, RQ5 |
| table9b_usfg_decileZ_nrc | same USFG statistic recomputed on decileZ(NRC), the fully-corrected metric | RQ1, RQ2, RQ5 |
| table10_metric_orientation | p_null diagnostics + Random-K win rate at each metric-fix stage (prob/NRC/decileZ-NRC) | FB3 |
| table5b_budget_trend_test | Jonckheere-Terpstra / OLS trend test for H3 (does gating AUC increase with M) | RQ3 |
| table_master_summary | one row per (dataset, model): headline numbers | quick reference |

## 1. Imports & paths

In [18]:
import os
import json
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, norm, rankdata, mannwhitneyu, t as tdist

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

BASE = "../kaggle-output"
NB1_DIR = f"{BASE}/NB1-outputs"
NB2_RESULT = f"{BASE}/NB2-outputs/NB2-result"
NB3_RESULT = f"{BASE}/NB3-outputs/NB3-result"
# NB4 writes flat filenames into /kaggle/working; the sub-folder name below is just the
# convention used when downloading that output locally. NB4 now evaluates on test_natural,
# so accept either the new or the historical folder name rather than silently reading none.
NB4_DIR = next((p for p in [f"{BASE}/NB4-outputs/NB4-test_natural",
                            f"{BASE}/NB4-outputs/NB4-test_500",
                            f"{BASE}/NB4-outputs"] if os.path.isdir(p)),
               f"{BASE}/NB4-outputs/NB4-test_natural")
NB5_DIR = f"{BASE}/NB5-outputs"

OUT_DIR = "../result/table"
os.makedirs(OUT_DIR, exist_ok=True)

DATASETS = ["home_credit", "taiwan", "gmsc"]
UQ_MODELS = ["M2", "M3", "M4", "M5"]     # models with an uncertainty estimate
ALL_MODELS = ["M1"] + UQ_MODELS
METHODS = ["shap", "lime"]
VARIANTS = ["prob", "logit", "norm"]
K_LABELS = ["10%", "20%", "30%", "40%", "50%"]

print("Paths configured. Checking availability...")
for name, path in [("NB1", NB1_DIR), ("NB2-result", NB2_RESULT), ("NB3-result", NB3_RESULT),
                    ("NB4", NB4_DIR), ("NB5", NB5_DIR)]:
    status = "found" if os.path.isdir(path) else "MISSING"
    print(f"  {name:<12} {path}  [{status}]")


Paths configured. Checking availability...
  NB1          ../kaggle-output/NB1-outputs  [found]
  NB2-result   ../kaggle-output/NB2-outputs/NB2-result  [found]
  NB3-result   ../kaggle-output/NB3-outputs/NB3-result  [found]
  NB4          ../kaggle-output/NB4-outputs/NB4-test_natural  [found]
  NB5          ../kaggle-output/NB5-outputs  [found]


## 2. Loading helpers

In [19]:
def load_json(path):
    if not os.path.exists(path):
        print(f"  [MISSING] {path}")
        return None
    with open(path) as f:
        return json.load(f)


def load_uncertainty(model, ds, natural=False):
    model_lc = model.lower()
    folder = NB2_RESULT if model == "M1" else NB3_RESULT
    suffix = "_natural" if natural else ""
    path = f"{folder}/{model_lc}_uncertainty{suffix}_{ds}.csv"
    if not os.path.exists(path):
        print(f"  [MISSING] {path}")
        return None
    return pd.read_csv(path)


def load_summary(model, ds, natural=False):
    model_lc = model.lower()
    folder = NB2_RESULT if model == "M1" else NB3_RESULT
    suffix = "_natural" if natural else ""
    return load_json(f"{folder}/{model_lc}_summary_metrics{suffix}_{ds}.json")


def load_faithfulness(variant, method, ds):
    """Merges the M1 file (NB2-result) with the M2-M5 file (NB5-output); both share the same
    filename pattern and both key their payload by model label, so merging is a dict update.
    v7 schema: every model's payload now also carries "base_probs" and "p_null_samples" (both
    length n_samples), and for variant="norm" the comp_samples/comp_random_samples arrays can
    contain NaN (undefined NRC ratio where |base_probs - p_null_samples| <= 0.005, ~2-6% of
    samples) -- callers that read the "norm" variant must mask NaN out before any scipy
    correlation call, which propagates a single NaN to the whole result otherwise."""
    m1_part = load_json(f"{NB2_RESULT}/faithfulness_{variant}_{method}_{ds}.json") or {}
    rest_part = load_json(f"{NB5_DIR}/faithfulness_{variant}_{method}_{ds}.json") or {}
    merged = {}
    merged.update(m1_part)
    merged.update(rest_part)
    return merged


def load_stability(ds):
    """Returns dict: model -> {jaccard_shap, jaccard_lime, max_sensitivity_shap} (arrays)."""
    out = {}
    m1_path = f"{NB2_RESULT}/m1_stability_{ds}.npz"
    if os.path.exists(m1_path):
        d = np.load(m1_path)
        out["M1"] = {"jaccard_shap": d["jaccard_shap"], "jaccard_lime": d["jaccard_lime"],
                      "max_sensitivity_shap": d["max_sensitivity_shap"]}
    else:
        print(f"  [MISSING] {m1_path}")

    rest_path = f"{NB5_DIR}/stability_{ds}.npz"
    if os.path.exists(rest_path):
        d = np.load(rest_path)
        for model in UQ_MODELS:
            out[model] = {
                "jaccard_shap": d[f"{model}_jaccard_shap"], "jaccard_lime": d[f"{model}_jaccard_lime"],
                "max_sensitivity_shap": d[f"{model}_max_sensitivity_shap"],
            }
    else:
        print(f"  [MISSING] {rest_path}")
    return out


def benjamini_hochberg(p_values):
    """Benjamini & Hochberg (1995) step-up FDR correction."""
    p = np.asarray(p_values, float)
    n = len(p)
    if n == 0:
        return np.array([])
    order = np.argsort(p)
    out = np.zeros(n)
    cummin = np.inf
    for rank in range(n, 0, -1):
        i = order[rank - 1]
        cummin = min(cummin, p[i] * n / rank)
        out[i] = cummin
    return np.clip(out, 0, 1)


def benjamini_hochberg_nan_safe(p_values):
    """Same as benjamini_hochberg, but NaN entries (e.g. M2's structurally-constant
    epistemic_unc -- see table4's Structural_NA column, or a degenerate USFG stratum in table9)
    are dropped before ranking, so they cannot inflate the family size `n` used in the
    correction for the genuine tests. NaN positions stay NaN in the output and must NOT be read
    as "not significant" -- they were never a real test. v7 audit fix: every call site used to
    pass the raw (NaN/degenerate-containing) array straight into benjamini_hochberg, silently
    inflating n (table7: 72 instead of 54; table9: 96 instead of 66; table3/table3b: 24 instead
    of 18)."""
    p = np.asarray(p_values, float)
    out = np.full(len(p), np.nan)
    ok = np.isfinite(p)
    if ok.sum() == 0:
        return out
    out[ok] = benjamini_hochberg(p[ok])
    return out


def partial_spearman(x, y, z):
    """Partial Spearman correlation of (x, y) controlling for z: rank all three variables,
    standardize the ranks, regress out z from both x and y, then take the Pearson correlation of
    the residuals. Significance via a t-test with df = n - 3 (one control variable). LINEAR-rank
    control only -- kept for comparison against partial_spearman_spline below, which is now the
    official H1/H4 test (see its docstring for why: this linear version has a true false-positive
    rate of ~52-56%, not 5%, whenever the confound z is non-monotone in the variable it is built
    from, which is exactly p(1-p)'s relationship to p once p > 0.5 samples are present)."""
    rx, ry, rz = rankdata(x), rankdata(y), rankdata(z)
    if rx.std() < 1e-12 or ry.std() < 1e-12 or rz.std() < 1e-12:
        return float("nan"), float("nan")
    rx = (rx - rx.mean()) / rx.std()
    ry = (ry - ry.mean()) / ry.std()
    rz = (rz - rz.mean()) / rz.std()
    ex = rx - np.dot(rx, rz) / np.dot(rz, rz) * rz
    ey = ry - np.dot(ry, rz) / np.dot(rz, rz) * rz
    n = len(x)
    denom = np.linalg.norm(ex) * np.linalg.norm(ey)
    if denom < 1e-12:
        return 0.0, 1.0
    r = np.dot(ex, ey) / denom
    df = n - 3
    t_stat = r * np.sqrt(df / max(1e-12, 1 - r ** 2))
    p = 2 * tdist.sf(abs(t_stat), df)     # v7 audit fix: was 2*(1-tdist.cdf(...)), which
    return float(r), float(p)             # underflows to a printed p=0.000000 for |t| large


def spline_basis(v, dfree=8):
    """Truncated-cubic-power spline basis (dfree knots at quantiles of rank(v) rescaled to
    [0, 1], plus an intercept and a linear term) used by partial_spearman_spline to residualize
    a possibly NON-monotone relationship with the confound. A LINEAR-rank control only removes
    the monotone part of x~z and y~z, which understates control whenever the confound itself is
    non-monotone in the raw variable it is built from -- p(1-p) is exactly this: non-monotone in
    p once p > 0.5 samples are present (Taiwan has ~18% of them)."""
    r = rankdata(v)
    u = (r - r.min()) / (r.max() - r.min())
    knots = np.quantile(u, np.linspace(0, 1, dfree + 2)[1:-1])
    cols = [np.ones_like(u), u] + [np.clip(u - k, 0, None) ** 3 for k in knots]
    return np.column_stack(cols)


def _zrank(v):
    """Rank, then z-score. Returns an all-NaN array (not a crash / warning-laden 0/0) if v is
    (near-)constant."""
    r = rankdata(v)
    sd = r.std()
    if sd < 1e-12:
        return np.full(len(v), np.nan)
    return (r - r.mean()) / sd


def _rank_rows(M):
    """Ordinal rank (1..n_cols) along axis=1 of a 2-D array, fully vectorized. Used only inside
    the permutation null of partial_spearman_spline, where thousands of rows need ranking; the
    single OBSERVED statistic is still computed with exact scipy.stats.spearmanr."""
    order = np.argsort(M, axis=1, kind="mergesort")
    ranks = np.empty(M.shape, dtype=float)
    base = np.arange(1, M.shape[1] + 1, dtype=float)
    np.put_along_axis(ranks, order, np.tile(base, (M.shape[0], 1)), axis=1)
    return ranks


def partial_spearman_spline(x, y, z, dfree=8, n_perm=3000, n_bins=20, seed=0):
    """Partial Spearman of (x, y) controlling for z via SPLINE residualization (dfree knots on
    rank(z)) instead of partial_spearman's linear-rank control, plus a CONDITIONAL permutation
    p-value (y shuffled WITHIN n_bins strata of z, so the marginal x~z / y~z relationships
    survive under the null and only x _|_ y | z is tested) instead of a t-distribution, which
    assumes something the spline step does not guarantee. This is now the OFFICIAL H1/H4 test
    (table7). v7 audit fix: Monte-Carlo calibration (n=500, B=500-1000 replicates; see
    nb6fix_stat_helpers.py in the audit scratch dir) shows the linear-rank+t-test combination
    above has a TRUE false-positive rate of ~52-56%, not the nominal 5%, under a non-monotone
    confound shaped like p(1-p) with p > 0.5 samples present -- exactly Taiwan's regime. The
    spline + conditional-permutation combination stays calibrated at ~4-5% under the same
    synthetic DGPs while keeping full power to detect a genuine partial correlation.
    Returns (nan, nan) if x, y or z is (near-)constant (e.g. M2's structurally-zero
    epistemic_unc) or if fewer than 20 samples remain -- there is nothing to test."""
    x, y, z = np.asarray(x, float), np.asarray(y, float), np.asarray(z, float)
    n = len(x)
    if n < 20 or np.nanstd(x) < 1e-12 or np.nanstd(y) < 1e-12 or np.nanstd(z) < 1e-12:
        return float("nan"), float("nan")

    X = spline_basis(z, dfree)
    Xpinv = np.linalg.pinv(X)
    zx, zy = _zrank(x), _zrank(y)
    ex = zx - X @ (Xpinv @ zx)
    ey_obs = zy - X @ (Xpinv @ zy)
    obs = spearmanr(ex, ey_obs).statistic
    if not np.isfinite(obs):
        return float("nan"), float("nan")

    rng = np.random.RandomState(seed)
    order = np.argsort(z)
    bins = [b for b in np.array_split(order, n_bins) if len(b) > 1]

    idx_mat = np.tile(np.arange(n), (n_perm, 1))
    for idx in bins:
        k = len(idx)
        keys = rng.random((n_perm, k))
        idx_mat[:, idx] = idx[np.argsort(keys, axis=1)]
    Yp = y[idx_mat]                                     # (n_perm, n): y shuffled within strata

    ranks = _rank_rows(Yp)
    rsd = ranks.std(axis=1, keepdims=True)
    rsd[rsd < 1e-12] = np.nan
    Zyp = (ranks - ranks.mean(axis=1, keepdims=True)) / rsd

    B_all = (Zyp @ X) @ np.linalg.pinv(X.T @ X)          # (n_perm, p) coefficients, batched
    Eyp = Zyp - B_all @ X.T                              # (n_perm, n) residuals, batched

    r_ex = rankdata(ex)
    R = _rank_rows(Eyp)
    r_ex_c = r_ex - r_ex.mean()
    Rc = R - R.mean(axis=1, keepdims=True)
    num = Rc @ r_ex_c
    den = np.sqrt((Rc ** 2).sum(axis=1) * (r_ex_c ** 2).sum())
    perm_rhos = num / np.maximum(den, 1e-12)

    p = (np.sum(np.abs(perm_rhos) >= abs(obs) - 1e-9) + 1) / (n_perm + 1)
    return float(obs), float(p)


def decile_z(x, v, n_bins=10):
    """Z-score x WITHIN each decile bin of v (bins defined by quantiles of v). Used to turn the
    per-sample NRC (the sign-corrected, normalized Comprehensiveness -- the "norm" faithfulness
    variant) into a metric with a locally-constant null distribution across the full range of
    v = p*(1-p), instead of one whose scale itself depends on where a sample sits along the
    confound. Of a 9-candidate comparison run during the audit, this was the best-calibrated
    (near-zero null bias, highest SNR on an injected-gap synthetic check -- independently
    replicated for this notebook in nb6fix_stat_helpers.py TEST 6, audit scratch dir).
    Returns NaN wherever x or v is non-finite, or a bin has fewer than 5 finite samples."""
    x = np.asarray(x, float)
    v = np.asarray(v, float)
    out = np.full(len(x), np.nan)
    ok = np.isfinite(x) & np.isfinite(v)
    if ok.sum() < 50:
        return out
    edges = np.quantile(v[ok], np.linspace(0, 1, n_bins + 1))
    edges[-1] += 1e-9
    b = np.clip(np.digitize(v, edges) - 1, 0, n_bins - 1)
    for i in range(n_bins):
        m = (b == i) & ok
        if m.sum() >= 5:
            out[m] = (x[m] - x[m].mean()) / max(x[m].std(), 1e-9)
    return out


def interaction_test(epi, comp, q, y):
    """H4, done as an INTERACTION test instead of two separate Fisher-Z-compared correlations:
    rank(Comp) ~ 1 + rank(epi) + rank(q) + y + rank(epi)*y + rank(q)*y, OLS on standardized
    ranks. Returns (coef, se, t, p) for the rank(epi)*y term -- does the epistemic/Comp
    relationship genuinely differ by class, after controlling for q = p(1-p) main effects AND a
    q*y interaction? v7 audit fix: table7 already shows the RAW epi~Comp correlation is largely
    a q-confound artifact, so table3b's old approach (Fisher-Z between two raw, uncontrolled
    minority/majority correlations) could not tell a genuine minority/majority difference in the
    epi->Comp relationship apart from a minority/majority difference in the q distribution.
    y must be 0/1. Returns all-NaN if epi is (near-)constant (M2) or n < 30."""
    epi, comp, q, y = (np.asarray(a, float) for a in (epi, comp, q, y))
    n = len(epi)
    if n < 30 or np.nanstd(epi) < 1e-12:
        return float("nan"), float("nan"), float("nan"), float("nan")
    E, Q, C = _zrank(epi), _zrank(q), _zrank(comp)
    X = np.column_stack([np.ones(n), E, Q, y, E * y, Q * y])
    b, *_ = np.linalg.lstsq(X, C, rcond=None)
    res = C - X @ b
    dof = n - X.shape[1]
    s2 = res @ res / dof
    cov = s2 * np.linalg.inv(X.T @ X)
    se = np.sqrt(np.diag(cov))
    tstat = b[4] / se[4]
    p = 2 * tdist.sf(abs(tstat), dof)
    return float(b[4]), float(se[4]), float(tstat), float(p)


def jonckheere_terpstra(groups):
    """Jonckheere-Terpstra trend test: H0 = no ordering effect across the (already-ordered)
    groups, H1 = stochastically increasing. Returns (J statistic, z, one-sided p). scipy has no
    built-in implementation."""
    k = len(groups)
    J = 0.0
    for i in range(k - 1):
        for j in range(i + 1, k):
            a = np.asarray(groups[i])[:, None]
            b = np.asarray(groups[j])[None, :]
            J += np.sum(a < b) + 0.5 * np.sum(a == b)
    ns = np.array([len(g) for g in groups])
    N = ns.sum()
    mu = (N ** 2 - np.sum(ns ** 2)) / 4
    var = (N ** 2 * (2 * N + 3) - np.sum(ns ** 2 * (2 * ns + 3))) / 72
    if var <= 0:
        return J, float("nan"), float("nan")
    z = (J - mu) / np.sqrt(var)
    return float(J), float(z), float(norm.sf(z))


def ols_trend_p(x, y):
    """OLS slope of y on x, two-sided t-test p-value for the slope (via tdist.sf, not
    1-tdist.cdf -- item 9). Used as the H3 budget-grid trend test, on log2(M): a direct
    dose-response regression, and/or a fallback when Jonckheere-Terpstra needs raw per-repeat
    draws that are not available."""
    X = np.column_stack([np.ones_like(x), x])
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    res = y - X @ b
    dof = len(y) - 2
    s2 = res @ res / dof
    se = np.sqrt(s2 * np.linalg.inv(X.T @ X)[1, 1])
    tstat = b[1] / se
    p = 2 * tdist.sf(abs(tstat), dof)
    return float(b[1]), float(se), float(tstat), float(p)


def fisher_z_spearman(rho1, rho2, n1, n2):
    """Fisher Z test for two independent Spearman correlations. The standard error uses the
    1.06 correction factor for Spearman's rho (Fieller, Hartley & Pearson, 1957); using the plain
    Pearson SE formula here understates the SE and inflates significance. Kept for comparison in
    table3b (raw, uncontrolled minority/majority gap) -- see interaction_test above for the
    official, q-controlled H4 test."""
    z1 = np.arctanh(np.clip(rho1, -0.9999, 0.9999))
    z2 = np.arctanh(np.clip(rho2, -0.9999, 0.9999))
    se = np.sqrt(1.06 / (n1 - 3) + 1.06 / (n2 - 3))
    z = (z1 - z2) / se
    p = 2 * norm.sf(abs(z))     # v7 audit fix: was 2*(1-norm.cdf(...)) -- underflow-prone
    return float(z), float(p)


print("Helpers defined (incl. v7 audit additions: spline partial correlation + conditional "
      "permutation, decile-Z normalization, H4 interaction test, Jonckheere-Terpstra / OLS "
      "trend test, NaN-safe FDR).")

Helpers defined (incl. v7 audit additions: spline partial correlation + conditional permutation, decile-Z normalization, H4 interaction test, Jonckheere-Terpstra / OLS trend test, NaN-safe FDR).


## 3. Table 1 — model performance

In [20]:
rows = []
for ds in DATASETS:
    for model in ALL_MODELS:
        for split, natural in [("balanced", False), ("natural", True)]:
            s = load_summary(model, ds, natural=natural)
            if s is None:
                continue
            row = {
                "Dataset": ds.upper().replace("_", " "), "Model": model, "Split": split,
                "AUC": s["predictive"]["auc"], "AUPRC": s["predictive"]["auprc"],
                "ECE": s["calibration"]["ece"], "NLL": s["calibration"]["nll"],
                "Brier": s["calibration"]["brier"], "N": s.get("n_samples"),
                "Prevalence": s.get("prevalence"),
            }
            ud = s.get("uncertainty_decomposition")
            if ud is not None:
                row.update({
                    "Mean_Aleatoric": ud["mean_aleatoric"], "Mean_Intra": ud["mean_intra"],
                    "Mean_Inter": ud["mean_inter"], "Mean_Epistemic": ud["mean_epistemic"],
                    "Mean_Total": ud["mean_total"], "Epistemic_Pct": ud["epistemic_pct_of_total"],
                })
            rows.append(row)

table1 = pd.DataFrame(rows)
table1.to_csv(f"{OUT_DIR}/table1_model_performance.csv", index=False)
print(f"table1_model_performance.csv  ({len(table1)} rows)")
print(table1[table1["Split"] == "balanced"].to_string(index=False))


table1_model_performance.csv  (30 rows)
    Dataset Model    Split      AUC    AUPRC      ECE      NLL    Brier    N  Prevalence  Mean_Aleatoric  Mean_Intra  Mean_Inter  Mean_Epistemic  Mean_Total  Epistemic_Pct
HOME CREDIT    M1 balanced 0.737654 0.727568 0.385534 1.159131 0.370354 2000         0.5             NaN         NaN         NaN             NaN         NaN            NaN
HOME CREDIT    M2 balanced 0.725350 0.721898 0.393441 1.167150 0.379845 2000         0.5        0.300015    0.000000    0.000000        0.000000    0.300015       0.000000
HOME CREDIT    M3 balanced 0.726107 0.723081 0.394120 1.162568 0.380920 2000         0.5        0.298807    0.003223    0.000000        0.003223    0.302030       1.281644
HOME CREDIT    M4 balanced 0.728987 0.724895 0.388622 1.145093 0.374651 2000         0.5        0.306621    0.000000    0.001781        0.001781    0.308402       0.619564
HOME CREDIT    M5 balanced 0.728529 0.724782 0.386773 1.135193 0.373244 2000         0.5        0.30

## 4. Table 2 — faithfulness (3 variants x SHAP/LIME x K grid)

In [21]:
rows = []
for ds in DATASETS:
    for variant in VARIANTS:
        for method in METHODS:
            payload = load_faithfulness(variant, method, ds)
            for model, d in payload.items():
                for k_str in d["comp_mean"]:
                    rows.append({
                        "Dataset": ds.upper().replace("_", " "), "Variant": variant,
                        "Method": method.upper(), "Model": model, "K": int(k_str),
                        "Comp": d["comp_mean"][k_str], "Suff": d["suff_mean"][k_str],
                        "CompRandom": d["comp_random_mean"][k_str],
                        "SuffRandom": d["suff_random_mean"][k_str],
                        "Comp_beats_random": d["comp_mean"][k_str] > d["comp_random_mean"][k_str],
                    })

table2 = pd.DataFrame(rows)
table2.to_csv(f"{OUT_DIR}/table2_faithfulness.csv", index=False)
print(f"table2_faithfulness.csv  ({len(table2)} rows)")

win_rate = table2.groupby(["Dataset", "Variant", "Method"])["Comp_beats_random"].mean()
print("\nRandom-K control win rate (Comp_XAI > Comp_random), by dataset/variant/method:")
print(win_rate.to_string())


table2_faithfulness.csv  (450 rows)

Random-K control win rate (Comp_XAI > Comp_random), by dataset/variant/method:
Dataset      Variant  Method
GMSC         logit    LIME      1.0
                      SHAP      1.0
             norm     LIME      1.0
                      SHAP      1.0
             prob     LIME      1.0
                      SHAP      1.0
HOME CREDIT  logit    LIME      1.0
                      SHAP      1.0
             norm     LIME      1.0
                      SHAP      1.0
             prob     LIME      1.0
                      SHAP      1.0
TAIWAN       logit    LIME      1.0
                      SHAP      1.0
             norm     LIME      1.0
                      SHAP      1.0
             prob     LIME      1.0
                      SHAP      1.0


## 5. Table 2b — stability (reproducibility + local robustness)

In [22]:
rows = []
for ds in DATASETS:
    stab = load_stability(ds)
    for model, d in stab.items():
        rows.append({
            "Dataset": ds.upper().replace("_", " "), "Model": model,
            "Jaccard_SHAP": float(np.mean(d["jaccard_shap"])),
            "Jaccard_LIME": float(np.mean(d["jaccard_lime"])),
            "MaxSensitivity_SHAP": float(np.mean(d["max_sensitivity_shap"])),
        })

table2b = pd.DataFrame(rows)
table2b.to_csv(f"{OUT_DIR}/table2b_stability.csv", index=False)
print(f"table2b_stability.csv  ({len(table2b)} rows)")
print(table2b.to_string(index=False))


table2b_stability.csv  (15 rows)
    Dataset Model  Jaccard_SHAP  Jaccard_LIME  MaxSensitivity_SHAP
HOME CREDIT    M1      1.000000      0.261356             0.605820
HOME CREDIT    M2      0.737408      0.344454             0.402359
HOME CREDIT    M3      0.737408      0.344454             0.402359
HOME CREDIT    M4      0.743558      0.379454             0.287848
HOME CREDIT    M5      0.736732      0.365118             0.316196
     TAIWAN    M1      1.000000      0.430265             1.203636
     TAIWAN    M2      0.825701      0.524083             0.255216
     TAIWAN    M3      0.825701      0.524083             0.255216
     TAIWAN    M4      0.809118      0.506358             0.236090
     TAIWAN    M5      0.811706      0.541433             0.247482
       GMSC    M1      1.000000      0.993333             1.435975
       GMSC    M2      0.957037      0.869630             0.587434
       GMSC    M3      0.957037      0.869630             0.587434
       GMSC    M4      0.9688

## 6. Table 3 — Spearman correlation of each uncertainty component with faithfulness

Computed at the median K (30%), `prob` variant, split into All / Minority (default) / Majority
(non-default). Covers `epistemic`, `aleatoric`, `intra`, and `inter` in one pass: the aleatoric
column is the negative-control (should show weak/no signal if the epistemic signal is genuine),
and comparing `intra` vs `inter` directly addresses RQ2/H2 (does cross-basin disagreement carry
more gating signal than within-basin dropout variance).


In [23]:
UNC_COLS = ["epistemic_unc", "aleatoric_unc", "intra_unc", "inter_unc"]

rows = []
for ds in DATASETS:
    for model in UQ_MODELS:
        unc = load_uncertainty(model, ds, natural=False)
        if unc is None:
            continue
        for method in METHODS:
            payload = load_faithfulness("prob", method, ds)
            if model not in payload:
                continue
            k_list = payload[model]["k_list"]
            mid_k = str(k_list[len(k_list) // 2])
            comp = np.array(payload[model]["comp_samples"][mid_k])
            n = min(len(unc), len(comp))
            u = unc.iloc[:n].reset_index(drop=True)
            c = comp[:n]

            for group_name, mask in [("All", np.ones(n, dtype=bool)),
                                      ("Minority", (u["y_true"] == 1).values),
                                      ("Majority", (u["y_true"] == 0).values)]:
                if mask.sum() < 5:
                    continue
                row = {"Dataset": ds.upper().replace("_", " "), "Model": model,
                       "Method": method.upper(), "Group": group_name, "N": int(mask.sum()),
                       "K": int(mid_k)}
                for col in UNC_COLS:
                    if col not in u.columns:
                        continue
                    rho, p = spearmanr(u[col].values[mask], c[mask])
                    row[f"rho_{col.replace('_unc', '')}"] = rho
                    row[f"p_{col.replace('_unc', '')}"] = p
                rows.append(row)

table3 = pd.DataFrame(rows)
if "p_epistemic" in table3.columns:
    all_mask = table3["Group"] == "All"
    table3.loc[all_mask, "p_epistemic_FDR"] = benjamini_hochberg_nan_safe(table3.loc[all_mask, "p_epistemic"].values)
    table3.loc[all_mask, "Sig_epistemic_FDR"] = table3.loc[all_mask, "p_epistemic_FDR"] < 0.05
    # v7 audit fix (item 5): declare the FDR family size explicitly (FB6a) -- M2 rows are
    # excluded from the count because epistemic_unc is a structural constant there (see table4).
    n_fam3 = int(np.isfinite(table3.loc[all_mask, "p_epistemic"].values).sum())
    print(f"FDR family (table3, p_epistemic, Group=All): {n_fam3}/{int(all_mask.sum())} tests "
          f"actually corrected (the rest are M2's structurally-undefined epistemic_unc).")

table3.to_csv(f"{OUT_DIR}/table3_spearman_uncertainty_components.csv", index=False)
print(f"table3_spearman_uncertainty_components.csv  ({len(table3)} rows)")
print(table3[table3["Group"] == "All"][["Dataset", "Model", "Method", "rho_epistemic", "rho_aleatoric", "rho_intra", "rho_inter"]].to_string(index=False))

C:\Users\User\AppData\Local\Temp\ipykernel_18176\3049969804.py:31: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, p = spearmanr(u[col].values[mask], c[mask])
C:\Users\User\AppData\Local\Temp\ipykernel_18176\3049969804.py:31: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, p = spearmanr(u[col].values[mask], c[mask])
C:\Users\User\AppData\Local\Temp\ipykernel_18176\3049969804.py:31: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, p = spearmanr(u[col].values[mask], c[mask])
C:\Users\User\AppData\Local\Temp\ipykernel_18176\3049969804.py:31: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, p = spearmanr(u[col].values[mask], c[mask])
C:\Users\User\AppData\Local\Temp\ipykernel_18176\3049969804.py:31: ConstantInputWarning: An input array is constant; the correlation coefficient is not 

FDR family (table3, p_epistemic, Group=All): 18/24 tests actually corrected (the rest are M2's structurally-undefined epistemic_unc).
table3_spearman_uncertainty_components.csv  (72 rows)
    Dataset Model Method  rho_epistemic  rho_aleatoric  rho_intra  rho_inter
HOME CREDIT    M2   SHAP            NaN       0.766444        NaN        NaN
HOME CREDIT    M2   LIME            NaN       0.685703        NaN        NaN
HOME CREDIT    M3   SHAP       0.412546       0.766005   0.412546        NaN
HOME CREDIT    M3   LIME       0.393206       0.686657   0.393206        NaN
HOME CREDIT    M4   SHAP       0.486141       0.761719        NaN   0.486141
HOME CREDIT    M4   LIME       0.469913       0.707757        NaN   0.469913
HOME CREDIT    M5   SHAP       0.416759       0.759271   0.395659   0.325552
HOME CREDIT    M5   LIME       0.423027       0.726406   0.403764   0.327177
     TAIWAN    M2   SHAP            NaN       0.707840        NaN        NaN
     TAIWAN    M2   LIME            NaN   

## 7. Table 3b — Fisher Z, minority vs majority (corrected SE)

In [24]:
rows = []
for ds in DATASETS:
    for model in UQ_MODELS:
        sub_min = table3[(table3["Dataset"] == ds.upper().replace("_", " ")) &
                          (table3["Model"] == model) & (table3["Group"] == "Minority")]
        sub_maj = table3[(table3["Dataset"] == ds.upper().replace("_", " ")) &
                          (table3["Model"] == model) & (table3["Group"] == "Majority")]

        unc = load_uncertainty(model, ds, natural=False)
        epi_full = unc["epistemic_unc"].values if unc is not None else None
        pbar_full = unc["pred_mean"].values if unc is not None else None
        y_full = unc["y_true"].values.astype(float) if unc is not None else None

        for method in ["SHAP", "LIME"]:
            r_min = sub_min[sub_min["Method"] == method]
            r_maj = sub_maj[sub_maj["Method"] == method]
            if r_min.empty or r_maj.empty:
                continue
            n_min, n_maj = int(r_min["N"].values[0]), int(r_maj["N"].values[0])
            rho_min, rho_maj = r_min["rho_epistemic"].values[0], r_maj["rho_epistemic"].values[0]
            z, p = fisher_z_spearman(rho_min, rho_maj, n_min, n_maj)

            row = {"Dataset": ds.upper().replace("_", " "), "Model": model, "Method": method,
                   "rho_minority": rho_min, "rho_majority": rho_maj,
                   "Fisher_Z": z, "P_value": p}

            # v7 audit addition (item 3, H4): an INTERACTION test on the raw per-sample data,
            # instead of comparing two raw (q-confounded) correlations -- table7 already shows
            # the raw epi~Comp correlation is itself largely a p(1-p) confound artifact, so a
            # minority/majority gap in the RAW correlations does not by itself demonstrate a
            # genuine minority/majority difference in the epi->Comp relationship. This is now
            # the official H4 answer; the Fisher-Z columns above are kept only for comparison.
            payload = load_faithfulness("prob", method.lower(), ds)
            if unc is not None and model in payload:
                k_list = payload[model]["k_list"]
                mid_k = str(k_list[len(k_list) // 2])
                comp = np.array(payload[model]["comp_samples"][mid_k])
                n = min(len(unc), len(comp))
                e, c, pb, yy = epi_full[:n], comp[:n], pbar_full[:n], y_full[:n]
                q = pb * (1 - pb)
                coef, se, tstat, p_int = interaction_test(e, c, q, yy)
                row.update({"interaction_coef": coef, "interaction_se": se,
                            "interaction_t": tstat, "interaction_p": p_int, "N_interaction": n})
            rows.append(row)

table3b = pd.DataFrame(rows)
if len(table3b):
    table3b["P_FDR"] = benjamini_hochberg_nan_safe(table3b["P_value"].values)
    table3b["Sig_FDR"] = table3b["P_FDR"] < 0.05
    n_fam_fz = int(np.isfinite(table3b["P_value"].values).sum())
    print(f"FDR family (table3b, raw Fisher-Z minority-vs-majority): {n_fam_fz}/{len(table3b)} "
          f"tests (M2 excluded -- degenerate epistemic_unc).")
    if "interaction_p" in table3b.columns:
        table3b["interaction_p_FDR"] = benjamini_hochberg_nan_safe(table3b["interaction_p"].values)
        table3b["interaction_Sig_FDR"] = table3b["interaction_p_FDR"] < 0.05
        n_fam_int = int(np.isfinite(table3b["interaction_p"].values).sum())
        print(f"FDR family (table3b, epi x minority-class INTERACTION test): "
              f"{n_fam_int}/{len(table3b)} tests.")

table3b.to_csv(f"{OUT_DIR}/table3b_fisher_z_minority_majority.csv", index=False)
print(f"table3b_fisher_z_minority_majority.csv  ({len(table3b)} rows)")
print(table3b.to_string(index=False))
print(f"\nSignificant RAW asymmetries (Fisher-Z, FDR<0.05 -- answers the q-CONFOUNDED question, "
      f"kept only for comparison): {int(table3b['Sig_FDR'].sum()) if len(table3b) else 0}/{len(table3b)}")
if "interaction_Sig_FDR" in table3b.columns:
    print(f"Significant epi x class INTERACTION (FDR<0.05 -- the q-controlled, official H4 "
          f"test): {int(table3b['interaction_Sig_FDR'].sum())}/{len(table3b)}")

FDR family (table3b, raw Fisher-Z minority-vs-majority): 18/24 tests (M2 excluded -- degenerate epistemic_unc).
FDR family (table3b, epi x minority-class INTERACTION test): 18/24 tests.
table3b_fisher_z_minority_majority.csv  (24 rows)
    Dataset Model Method  rho_minority  rho_majority  Fisher_Z      P_value  interaction_coef  interaction_se  interaction_t  interaction_p  N_interaction        P_FDR  Sig_FDR  interaction_p_FDR  interaction_Sig_FDR
HOME CREDIT    M2   SHAP           NaN           NaN       NaN          NaN               NaN             NaN            NaN            NaN           2000          NaN    False                NaN                False
HOME CREDIT    M2   LIME           NaN           NaN       NaN          NaN               NaN             NaN            NaN            NaN           2000          NaN    False                NaN                False
HOME CREDIT    M3   SHAP      0.314346      0.414521 -2.509000 1.210735e-02         -0.023689        0.035696    

## 8. Table 4 — gating metrics

Standard Gating AUC (`AUC(1[misclassified], uncertainty)`), class-discriminative AUC
(`AUC(y, uncertainty)`, the v6 definition — kept for continuity, explicitly relabeled), and AURC,
computed directly from each canonical model's uncertainty CSV.


In [25]:
from sklearn.metrics import roc_auc_score


def compute_aurc(uncertainty, errors):
    order = np.argsort(uncertainty)
    sorted_errors = errors[order]
    n = len(errors)
    cum_errors = np.cumsum(sorted_errors)
    coverage = np.arange(1, n + 1) / n
    risk = cum_errors / np.arange(1, n + 1)
    # np.trapz was removed in NumPy 2.2+ (renamed np.trapezoid) -- use a manual, version-
    # independent trapezoidal rule instead of depending on either name existing.
    return float(np.sum((risk[1:] + risk[:-1]) / 2.0 * np.diff(coverage)))


rows = []
for ds in DATASETS:
    for model in UQ_MODELS:
        unc = load_uncertainty(model, ds, natural=True)   # deployment-realistic split
        if unc is None:
            continue
        y_true = unc["y_true"].values
        pred = unc["pred_mean"].values
        epi = unc["epistemic_unc"].values
        ale = unc["aleatoric_unc"].values
        # base-rate operating point: predict the top-`prevalence` fraction positive,
        # instead of a naive 0.5 cut on a model trained on an imbalanced distribution.
        prevalence = y_true.mean()
        tau = np.quantile(pred, 1.0 - prevalence)
        y_pred = (pred >= tau).astype(int)
        misclassified = (y_pred != y_true).astype(int)
        # v7 audit addition (item 2, H2): a cheap, UQ-free "confidence margin" baseline --
        # distance from the operating threshold -- defined for every model, including M2.
        margin = -np.abs(pred - tau)

        # v7 audit addition (item 2, H2): epistemic_unc is IDENTICALLY 0 for M2 by construction
        # (a single basin, T=1 MC-Dropout pass -> no basin-to-basin disagreement, no dropout
        # averaging), so its Gating_AUC = 0.500 / AURC = error-rate are DEFINITIONAL
        # consequences of a constant score, not an empirical finding about "uncertainty carries
        # no gating signal for M2". The margin/aleatoric baselines below are non-degenerate for
        # M2 and quantify what a cheap, UQ-free routing score already gets you there.
        structural_na = bool(np.ptp(epi) < 1e-12)

        gating_std = np.nan if structural_na else (
            roc_auc_score(misclassified, epi) if misclassified.sum() > 0 else np.nan)
        gating_cd = np.nan if structural_na else roc_auc_score(y_true, epi)
        aurc = np.nan if structural_na else compute_aurc(epi, misclassified)

        gating_std_margin = roc_auc_score(misclassified, margin) if misclassified.sum() > 0 else np.nan
        gating_cd_margin = roc_auc_score(y_true, margin)
        aurc_margin = compute_aurc(margin, misclassified)

        gating_std_ale = roc_auc_score(misclassified, ale) if misclassified.sum() > 0 else np.nan
        gating_cd_ale = roc_auc_score(y_true, ale)
        aurc_ale = compute_aurc(ale, misclassified)

        rows.append({
            "Dataset": ds.upper().replace("_", " "), "Model": model,
            "Gating_AUC_standard": gating_std, "Gating_AUC_class_discriminative": gating_cd,
            "AURC": aurc, "Error_rate": float(misclassified.mean()),
            "Structural_NA": structural_na,
            "Note": ("epistemic_unc is a structural constant (0) for M2 -- AUC=0.500/AURC="
                     "error-rate would hold BY DEFINITION here, not as an empirical result"
                     ) if structural_na else "",
            "Gating_AUC_standard_margin": gating_std_margin,
            "Gating_AUC_class_discriminative_margin": gating_cd_margin, "AURC_margin": aurc_margin,
            "Gating_AUC_standard_aleatoric": gating_std_ale,
            "Gating_AUC_class_discriminative_aleatoric": gating_cd_ale, "AURC_aleatoric": aurc_ale,
        })

table4 = pd.DataFrame(rows)
table4.to_csv(f"{OUT_DIR}/table4_gating_metrics.csv", index=False)
print(f"table4_gating_metrics.csv  ({len(table4)} rows)")
print(table4.to_string(index=False))
print("\nStructurally N/A rows (epistemic_unc constant by construction -- see Note column):")
print(table4[table4["Structural_NA"]][["Dataset", "Model"]].to_string(index=False))

table4_gating_metrics.csv  (12 rows)
    Dataset Model  Gating_AUC_standard  Gating_AUC_class_discriminative     AURC  Error_rate  Structural_NA                                                                                                                                     Note  Gating_AUC_standard_margin  Gating_AUC_class_discriminative_margin  AURC_margin  Gating_AUC_standard_aleatoric  Gating_AUC_class_discriminative_aleatoric  AURC_aleatoric
HOME CREDIT    M2                  NaN                              NaN      NaN       0.114           True epistemic_unc is a structural constant (0) for M2 -- AUC=0.500/AURC=error-rate would hold BY DEFINITION here, not as an empirical result                    0.733610                                0.629790     0.057967                       0.816935                                   0.719364        0.040076
HOME CREDIT    M3             0.708099                         0.621777 0.058434       0.113          False                        

## 9. Table 5 — budget grid summary (RQ3)

In [26]:
dfs = []
for ds in DATASETS:
    path = f"{NB4_DIR}/budget_grid_summary_{ds}.csv"
    if os.path.exists(path):
        dfs.append(pd.read_csv(path))
    else:
        print(f"  [MISSING] {path}")

table5 = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
if len(table5):
    table5["dataset"] = table5["dataset"].str.upper().str.replace("_", " ")
table5.to_csv(f"{OUT_DIR}/table5_budget_grid_summary.csv", index=False)
print(f"table5_budget_grid_summary.csv  ({len(table5)} rows)")
if len(table5):
    print(table5[table5["metric"] == "gating_auc_std"].to_string(index=False))


table5_budget_grid_summary.csv  (144 rows)
    dataset  config_id  M  T         metric     mean   ci_low  ci_high  n_repeats
HOME CREDIT          0  1 50 gating_auc_std 0.678327 0.602860 0.712552         30
HOME CREDIT          1  2 25 gating_auc_std 0.684304 0.651939 0.713315         30
HOME CREDIT          2  5 10 gating_auc_std 0.692686 0.651546 0.733257         30
HOME CREDIT          3 10  5 gating_auc_std 0.696249 0.678041 0.713419         30
HOME CREDIT          4 25  2 gating_auc_std 0.704670 0.688458 0.714289         30
HOME CREDIT          5 50  1 gating_auc_std 0.726288 0.726288 0.726288          1
     TAIWAN          0  1 50 gating_auc_std 0.590959 0.548044 0.611549         30
     TAIWAN          1  2 25 gating_auc_std 0.594524 0.563304 0.617466         30
     TAIWAN          2  5 10 gating_auc_std 0.601363 0.586476 0.618506         30
     TAIWAN          3 10  5 gating_auc_std 0.604462 0.583036 0.623492         30
     TAIWAN          4 25  2 gating_auc_std 0.606426 0.

## 9b. Table 5b — H3 trend test (does gating quality increase monotonically with M?)

v7 audit addition (item 4). The budget-grid table above was previously read by eye with no
significance test attached to "gating quality increases with ensemble size M". Jonckheere-Terpstra
(H1: stochastically increasing across the 6 ordered `M` configs) on the raw per-repeat draws
(`budget_grid_raw_{ds}.csv`, same NB4 output folder as table5's own source) when available, plus
an OLS trend of `gating_auc_std` on `log2(M)` either way (a direct dose-response slope, and the
fallback when only the per-config summary means can be found). Reported separately per dataset.


In [27]:
trend_rows = []
for ds in DATASETS:
    raw_path = f"{NB4_DIR}/budget_grid_raw_{ds}.csv"
    ds_label = ds.upper().replace("_", " ")
    row = {"Dataset": ds_label, "Metric": "gating_auc_std"}

    if os.path.exists(raw_path):
        raw = pd.read_csv(raw_path)
        row["Source"] = "raw_per_repeat"
        row["N_obs"] = len(raw)
        order_ids = raw.sort_values("M")["config_id"].unique()
        groups = [raw[raw["config_id"] == cid]["gating_auc_std"].dropna().values for cid in order_ids]
        Ms_sorted = [int(raw[raw["config_id"] == cid]["M"].iloc[0]) for cid in order_ids]
        row["M_order"] = str(Ms_sorted)
        if all(len(g) > 0 for g in groups):
            J, z, p_jt = jonckheere_terpstra(groups)
            row.update({"JT_statistic": J, "JT_z": z, "JT_p_onesided": p_jt})
        x_log2m = np.log2(raw["M"].values.astype(float))
        slope, se, tstat, p_ols = ols_trend_p(x_log2m, raw["gating_auc_std"].values)
        row.update({"OLS_slope_per_doubling": slope, "OLS_se": se, "OLS_t": tstat,
                    "OLS_p_twosided": p_ols})
    elif len(table5):
        sub = table5[(table5["dataset"] == ds_label) &
                      (table5["metric"] == "gating_auc_std")].sort_values("M")
        if len(sub) >= 3:
            row["Source"] = "summary_means_only"
            row["N_obs"] = len(sub)
            x_log2m = np.log2(sub["M"].values.astype(float))
            slope, se, tstat, p_ols = ols_trend_p(x_log2m, sub["mean"].values)
            row.update({"OLS_slope_per_doubling": slope, "OLS_se": se, "OLS_t": tstat,
                        "OLS_p_twosided": p_ols})
    trend_rows.append(row)

table5b = pd.DataFrame(trend_rows)
table5b.to_csv(f"{OUT_DIR}/table5b_budget_trend_test.csv", index=False)
print(f"table5b_budget_trend_test.csv  ({len(table5b)} rows)")
print(table5b.to_string(index=False))
if "JT_p_onesided" in table5b.columns:
    print("\nJonckheere-Terpstra H1='gating_auc_std stochastically increases with M', one-sided:")
    for _, r in table5b.dropna(subset=["JT_p_onesided"]).iterrows():
        print(f"  {r['Dataset']:14s} z={r['JT_z']:+.3f}  p={r['JT_p_onesided']:.4g}")

table5b_budget_trend_test.csv  (3 rows)
    Dataset         Metric         Source  N_obs               M_order  JT_statistic     JT_z  JT_p_onesided  OLS_slope_per_doubling   OLS_se    OLS_t  OLS_p_twosided
HOME CREDIT gating_auc_std raw_per_repeat    151 [1, 2, 5, 10, 25, 50]        6425.0 6.077837   6.090741e-10                0.005708 0.000909 6.282676    3.469958e-09
     TAIWAN gating_auc_std raw_per_repeat    151 [1, 2, 5, 10, 25, 50]        6007.0 4.704574   1.271983e-06                0.003586 0.000664 5.401432    2.559248e-07
       GMSC gating_auc_std raw_per_repeat    151 [1, 2, 5, 10, 25, 50]        4892.5 1.043088   1.484537e-01                0.003409 0.001895 1.798743    7.408354e-02

Jonckheere-Terpstra H1='gating_auc_std stochastically increases with M', one-sided:
  HOME CREDIT    z=+6.078  p=6.091e-10
  TAIWAN         z=+4.705  p=1.272e-06
  GMSC           z=+1.043  p=0.1485


## 10. Table 6 — uncertainty component distribution

In [28]:
rows = []
for ds in DATASETS:
    for model in UQ_MODELS:
        unc = load_uncertainty(model, ds, natural=False)
        if unc is None:
            continue
        for col in UNC_COLS:
            if col not in unc.columns:
                continue
            vals = unc[col].values
            rows.append({
                "Dataset": ds.upper().replace("_", " "), "Model": model,
                "Component": col.replace("_unc", ""), "Mean": float(np.mean(vals)),
                "Std": float(np.std(vals)), "CV": float(np.std(vals) / max(np.mean(vals), 1e-12)),
                "Min": float(np.min(vals)), "Max": float(np.max(vals)),
                "Median": float(np.median(vals)),
            })

table6 = pd.DataFrame(rows)
table6.to_csv(f"{OUT_DIR}/table6_uq_distribution.csv", index=False)
print(f"table6_uq_distribution.csv  ({len(table6)} rows)")
print(table6[table6["Component"] == "epistemic"].to_string(index=False))


table6_uq_distribution.csv  (48 rows)
    Dataset Model Component     Mean      Std       CV      Min      Max   Median
HOME CREDIT    M2 epistemic 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000
HOME CREDIT    M3 epistemic 0.003223 0.001592 0.494060 0.000432 0.015055 0.002876
HOME CREDIT    M4 epistemic 0.001781 0.001554 0.872362 0.000169 0.012586 0.001289
HOME CREDIT    M5 epistemic 0.004733 0.002840 0.599943 0.000719 0.023770 0.004024
     TAIWAN    M2 epistemic 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000
     TAIWAN    M3 epistemic 0.007395 0.004096 0.553801 0.001516 0.041259 0.006425
     TAIWAN    M4 epistemic 0.002359 0.001901 0.805514 0.000249 0.016883 0.001776
     TAIWAN    M5 epistemic 0.009221 0.004898 0.531186 0.001925 0.054647 0.008005
       GMSC    M2 epistemic 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000
       GMSC    M3 epistemic 0.003985 0.002824 0.708584 0.000321 0.022393 0.003310
       GMSC    M4 epistemic 0.001519 0.000955 0.628647 0.000

## 11. Table 7 — partial correlation (RQ1 + RQ4)

Core confound check. `rho_partial` controls for `p_bar * (1 - p_bar)`, the quantity both
epistemic uncertainty (via mutual information) and Comprehensiveness (via its upper bound) are
mechanically related to. `rho_aleatoric` is the negative control. Computed across all three
faithfulness variants (`prob`, `logit`, `norm`) so RQ4 (does the conclusion depend on which
faithfulness definition is used) can be answered directly from this one table.


In [29]:
rows = []
for ds in DATASETS:
    for model in UQ_MODELS:
        unc = load_uncertainty(model, ds, natural=False)
        if unc is None:
            continue
        epi = unc["epistemic_unc"].values
        ale = unc["aleatoric_unc"].values
        pbar = unc["pred_mean"].values
        pq = pbar * (1 - pbar)

        for variant in VARIANTS:
            for method in METHODS:
                payload = load_faithfulness(variant, method, ds)
                if model not in payload:
                    continue
                k_list = payload[model]["k_list"]
                mid_k = str(k_list[len(k_list) // 2])
                comp = np.array(payload[model]["comp_samples"][mid_k])
                n = min(len(epi), len(comp))
                e, a, q, c = epi[:n], ale[:n], pq[:n], comp[:n]

                # v7 audit fix (schema adaptation): the "norm" variant can now contain NaN
                # (undefined NRC ratio, |base_probs - p_null| <= 0.005, ~2-6% of samples).
                # scipy correlations propagate a single NaN to the WHOLE result, so it must be
                # masked out first. "prob"/"logit" are always fully dense (n_valid == n) --
                # this is a no-op there.
                ok = np.isfinite(c)
                n_valid = int(ok.sum())
                e, a, q, c = e[ok], a[ok], q[ok], c[ok]

                rho_raw, p_raw = spearmanr(e, c) if n_valid >= 10 else (np.nan, np.nan)
                rho_ale, p_ale = spearmanr(a, c) if n_valid >= 10 else (np.nan, np.nan)
                # v7 audit fix (item 1, the core H1/H4 fix): spline residualization + conditional
                # permutation p-value is now the OFFICIAL partial-correlation test (see
                # partial_spearman_spline's docstring). The original linear-rank + t-test result
                # is kept side by side as rho_partial_linear/p_partial_linear for comparison --
                # the before/after IS part of the audit's contribution.
                rho_partial, p_partial = partial_spearman_spline(e, c, q)
                rho_partial_lin, p_partial_lin = partial_spearman(e, c, q)

                shrink = ((1 - abs(rho_partial) / max(1e-9, abs(rho_raw))) * 100
                          if np.isfinite(rho_partial) and np.isfinite(rho_raw) else np.nan)
                degenerate = bool(np.ptp(epi) < 1e-12)
                rows.append({
                    "Dataset": ds.upper().replace("_", " "), "Model": model, "Method": method.upper(),
                    "Variant": variant, "N": n, "N_valid": n_valid,
                    "rho_raw": rho_raw, "p_raw": p_raw,
                    "rho_partial": rho_partial, "p_partial": p_partial,
                    "rho_partial_linear": rho_partial_lin, "p_partial_linear": p_partial_lin,
                    "rho_aleatoric": rho_ale, "p_aleatoric": p_ale,
                    "shrinkage_pct": shrink, "Degenerate": degenerate,
                })

table7 = pd.DataFrame(rows)
if len(table7):
    # v7 audit fix (item 5): NaN-safe FDR, family size declared explicitly (FB6a).
    table7["p_partial_FDR"] = benjamini_hochberg_nan_safe(table7["p_partial"].values)
    table7["Sig_partial_FDR"] = table7["p_partial_FDR"] < 0.05
    table7["p_partial_linear_FDR"] = benjamini_hochberg_nan_safe(table7["p_partial_linear"].values)
    table7["Sig_partial_linear_FDR"] = table7["p_partial_linear_FDR"] < 0.05
    n_fam7 = int(np.isfinite(table7["p_partial"].values).sum())
    print(f"FDR family (table7, p_partial -- spline+conditional-permutation, the OFFICIAL "
          f"H1/H4 test): {n_fam7}/{len(table7)} tests actually corrected (M2 rows excluded -- "
          f"degenerate epistemic_unc; previously all {len(table7)} rows incl. NaNs were passed "
          f"to the correction, inflating it).")
table7.to_csv(f"{OUT_DIR}/table7_partial_correlation.csv", index=False)
print(f"table7_partial_correlation.csv  ({len(table7)} rows)")

prob_view = table7[table7["Variant"] == "prob"]
print(prob_view[["Dataset", "Model", "Method", "rho_raw", "rho_partial", "rho_partial_linear",
                  "rho_aleatoric", "shrinkage_pct", "Sig_partial_FDR"]].to_string(index=False))
n_sig = int(table7["Sig_partial_FDR"].sum()) if len(table7) else 0
n_sig_lin = int(table7["Sig_partial_linear_FDR"].sum()) if len(table7) else 0
print(f"\nTotal (variant x method x model x dataset) combinations still significant "
      f"after controlling for p(1-p), spline+permutation (OFFICIAL): {n_sig}/{len(table7)}")
print(f"Same, using the OLD linear-rank+t-test control (kept for comparison only): "
      f"{n_sig_lin}/{len(table7)}")

C:\Users\User\AppData\Local\Temp\ipykernel_18176\541462791.py:32: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho_raw, p_raw = spearmanr(e, c) if n_valid >= 10 else (np.nan, np.nan)
C:\Users\User\AppData\Local\Temp\ipykernel_18176\541462791.py:32: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho_raw, p_raw = spearmanr(e, c) if n_valid >= 10 else (np.nan, np.nan)
C:\Users\User\AppData\Local\Temp\ipykernel_18176\541462791.py:32: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho_raw, p_raw = spearmanr(e, c) if n_valid >= 10 else (np.nan, np.nan)


FDR family (table7, p_partial -- spline+conditional-permutation, the OFFICIAL H1/H4 test): 54/72 tests actually corrected (M2 rows excluded -- degenerate epistemic_unc; previously all 72 rows incl. NaNs were passed to the correction, inflating it).
table7_partial_correlation.csv  (72 rows)
    Dataset Model Method  rho_raw  rho_partial  rho_partial_linear  rho_aleatoric  shrinkage_pct  Sig_partial_FDR
HOME CREDIT    M2   SHAP      NaN          NaN                 NaN       0.766444            NaN            False
HOME CREDIT    M2   LIME      NaN          NaN                 NaN       0.685703            NaN            False
HOME CREDIT    M3   SHAP 0.412546    -0.014832           -0.058903       0.766005      96.404651            False
HOME CREDIT    M3   LIME 0.393206     0.055005           -0.007479       0.686657      86.011107            False
HOME CREDIT    M4   SHAP 0.486141    -0.010070           -0.035774       0.761719      97.928665            False
HOME CREDIT    M4   LIME 

## 12. Table 8 — Spearman sensitivity across the full K grid

In [30]:
rows = []
for ds in DATASETS:
    for model in UQ_MODELS:
        unc = load_uncertainty(model, ds, natural=False)
        if unc is None:
            continue
        epi = unc["epistemic_unc"].values
        for method in METHODS:
            payload = load_faithfulness("prob", method, ds)
            if model not in payload:
                continue
            row = {"Dataset": ds.upper().replace("_", " "), "Model": model, "Method": method.upper()}
            vals = []
            for k_str in payload[model]["comp_samples"]:
                comp = np.array(payload[model]["comp_samples"][k_str])
                n = min(len(epi), len(comp))
                rho, _ = spearmanr(epi[:n], comp[:n])
                row[f"rho_K{k_str}"] = rho
                vals.append(rho)
            if vals:
                row["rho_min"] = min(vals)
                row["rho_max"] = max(vals)
                row["range"] = max(vals) - min(vals)
            rows.append(row)

table8 = pd.DataFrame(rows)
table8.to_csv(f"{OUT_DIR}/table8_spearman_sensitivity_K.csv", index=False)
print(f"table8_spearman_sensitivity_K.csv  ({len(table8)} rows)")
print(table8.to_string(index=False))
if "range" in table8.columns and len(table8):
    print(f"\nMean range across K: {table8['range'].mean():.4f}  Max range: {table8['range'].max():.4f}")


C:\Users\User\AppData\Local\Temp\ipykernel_18176\1192378486.py:17: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, _ = spearmanr(epi[:n], comp[:n])
C:\Users\User\AppData\Local\Temp\ipykernel_18176\1192378486.py:17: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, _ = spearmanr(epi[:n], comp[:n])
C:\Users\User\AppData\Local\Temp\ipykernel_18176\1192378486.py:17: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, _ = spearmanr(epi[:n], comp[:n])


table8_spearman_sensitivity_K.csv  (24 rows)
    Dataset Model Method  rho_K10  rho_K21  rho_K31  rho_K42  rho_K52  rho_min  rho_max    range   rho_K2   rho_K5   rho_K7   rho_K9  rho_K12   rho_K1   rho_K3   rho_K4
HOME CREDIT    M2   SHAP      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN
HOME CREDIT    M2   LIME      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN
HOME CREDIT    M3   SHAP 0.410503 0.429747 0.412546 0.402234 0.403063 0.402234 0.429747 0.027514      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN
HOME CREDIT    M3   LIME 0.417747 0.396705 0.393206 0.392669 0.377416 0.377416 0.417747 0.040331      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN
HOME CREDIT    M4   SHAP 0.504645 0.510985 0.486141 0.483205 0.485707 0.483205 0.510985 0.027780      NaN     

## 13. Table 9 — USFG with bootstrap CI + Mann-Whitney (RQ1, RQ2, RQ5)

Uncertainty-Stratified Faithfulness Gap: `mean(Comp | Q1 of uncertainty) - mean(Comp | Q4 of
uncertainty)`, normalized by the within-dataset standard deviation of Comp (replacing the v6
denominator `2|AUC-0.5|`, which was nearly constant within a dataset and had no clear theoretical
grounding). Computed for all four uncertainty components, so H1 (epistemic vs aleatoric) and H2
(intra vs inter) can both be read directly off this table. 95% CI via bootstrap resampling of the
two strata; Mann-Whitney U test for the strata difference, BH-FDR corrected.


In [31]:
def usfg_bootstrap(low_vals, high_vals, c_all, n_boot=1000, seed=42):
    """v7 audit fix (item 6): the previous version froze `denom` (the SD used to standardize the
    Q1-Q4 gap) at its point-estimate value across all bootstrap iterations, only resampling the
    numerator (low_vals, high_vals) -- understating the CI width, since denom is itself an
    estimate with its own sampling variability. Now c_all (the FULL array -- Comp, NRC or
    decileZ(NRC) -- the Q1/Q4 strata were carved from) is resampled too, and denom is recomputed
    from that resample on every iteration, alongside the numerator."""
    rng = np.random.RandomState(seed)
    denom0 = max(np.nanstd(c_all), 1e-9)
    point = (low_vals.mean() - high_vals.mean()) / denom0
    boots = np.empty(n_boot)
    for b in range(n_boot):
        bl = rng.choice(low_vals, size=len(low_vals), replace=True)
        bh = rng.choice(high_vals, size=len(high_vals), replace=True)
        bc = rng.choice(c_all, size=len(c_all), replace=True)
        d = max(np.nanstd(bc), 1e-9)
        boots[b] = (bl.mean() - bh.mean()) / d
    ci_lo, ci_hi = np.percentile(boots, [2.5, 97.5])
    return point, ci_lo, ci_hi


rows = []
for ds in DATASETS:
    for model in UQ_MODELS:
        unc = load_uncertainty(model, ds, natural=False)
        if unc is None:
            continue
        for method in METHODS:
            payload = load_faithfulness("prob", method, ds)
            if model not in payload:
                continue
            k_list = payload[model]["k_list"]
            mid_k = str(k_list[len(k_list) // 2])
            comp = np.array(payload[model]["comp_samples"][mid_k])
            n = min(len(unc), len(comp))
            c = comp[:n]

            for col in UNC_COLS:
                if col not in unc.columns:
                    continue
                u = unc[col].values[:n]

                # v7 audit fix (item 5): a (near-)constant unc column (M2's epistemic_unc,
                # M3's inter_unc, M4's intra_unc -- all structurally 0 by that model's own
                # decomposition) makes Q1==Q4==the WHOLE sample, so "low" and "high" become the
                # SAME n points and Mann-Whitney compares a sample to itself (p=1.0, n_Q1=n_Q4=n
                # -- not a real test, but previously still counted in the FDR family, inflating
                # it from 66 to 96). Emit an explicit NaN/Degenerate row instead.
                degenerate = bool(np.ptp(u) < 1e-12)
                if degenerate:
                    rows.append({
                        "Dataset": ds.upper().replace("_", " "), "Model": model,
                        "Method": method.upper(), "Component": col.replace("_unc", ""),
                        "USFG": np.nan, "CI95_low": np.nan, "CI95_high": np.nan,
                        "CI_excludes_0": False, "MannWhitney_p": np.nan, "rank_biserial": np.nan,
                        "n_Q1": len(u), "n_Q4": len(u), "Degenerate": True,
                    })
                    continue

                q1, q4 = np.percentile(u, 25), np.percentile(u, 75)
                low, high = c[u <= q1], c[u >= q4]
                if len(low) < 5 or len(high) < 5:
                    continue

                point, ci_lo, ci_hi = usfg_bootstrap(low, high, c)
                stat, p_mw = mannwhitneyu(low, high, alternative="two-sided")
                rank_biserial = 1 - 2 * stat / (len(low) * len(high))

                rows.append({
                    "Dataset": ds.upper().replace("_", " "), "Model": model, "Method": method.upper(),
                    "Component": col.replace("_unc", ""), "USFG": point,
                    "CI95_low": ci_lo, "CI95_high": ci_hi, "CI_excludes_0": bool(ci_lo * ci_hi > 0),
                    "MannWhitney_p": p_mw, "rank_biserial": rank_biserial,
                    "n_Q1": len(low), "n_Q4": len(high), "Degenerate": False,
                })

table9 = pd.DataFrame(rows)
if len(table9):
    table9["p_FDR"] = benjamini_hochberg_nan_safe(table9["MannWhitney_p"].values)
    table9["Sig_FDR"] = table9["p_FDR"] < 0.05
    n_fam9 = int(np.isfinite(table9["MannWhitney_p"].values).sum())
    print(f"FDR family (table9, MannWhitney_p): {n_fam9}/{len(table9)} tests actually corrected "
          f"({int(table9['Degenerate'].sum())} structurally-degenerate rows excluded).")
table9.to_csv(f"{OUT_DIR}/table9_usfg_bootstrap.csv", index=False)
print(f"table9_usfg_bootstrap.csv  ({len(table9)} rows)")

pivot = table9.pivot_table(index=["Dataset", "Model", "Method"], columns="Component", values="USFG")
print("\nUSFG by component (mean over methods shown where available):")
print(pivot.to_string())

FDR family (table9, MannWhitney_p): 66/96 tests actually corrected (30 structurally-degenerate rows excluded).
table9_usfg_bootstrap.csv  (96 rows)

USFG by component (mean over methods shown where available):
Component                 aleatoric  epistemic     inter     intra
Dataset     Model Method                                          
GMSC        M2    LIME    -1.822220        NaN       NaN       NaN
                  SHAP    -2.083301        NaN       NaN       NaN
            M3    LIME    -1.818698  -1.411459       NaN -1.411459
                  SHAP    -2.078995  -1.662020       NaN -1.662020
            M4    LIME    -1.832508  -1.160345 -1.160345       NaN
                  SHAP    -2.126829  -1.409617 -1.409617       NaN
            M5    LIME    -1.819940  -0.919658 -0.009459 -1.210159
                  SHAP    -2.114836  -1.129466 -0.097772 -1.424080
HOME CREDIT M2    LIME    -1.690793        NaN       NaN       NaN
                  SHAP    -1.874222        NaN       

## 13b. Table 9b — USFG recomputed on decileZ(NRC)

v7 audit addition (item 8). Table 9 above uses the raw `prob` faithfulness variant, unchanged, as
the historical/uncorrected reference (FB5: the before/after comparison is itself part of the
contribution — nothing is deleted). This table recomputes the same Q1-vs-Q4 USFG statistic on the
fully-corrected metric instead: the `norm` variant's per-sample NRC (signed, NaN-aware — see the
schema note in `load_faithfulness`), then z-scored within deciles of `p(1-p)` (`decile_z`, item 8's
namesake, "USFG-NRC" / "decileZ_NRC"). Same bootstrap fix (item 6) and NaN-safe FDR (item 5) as
table9.


In [32]:
rows = []
for ds in DATASETS:
    for model in UQ_MODELS:
        unc = load_uncertainty(model, ds, natural=False)
        if unc is None:
            continue
        for method in METHODS:
            payload = load_faithfulness("norm", method, ds)
            if model not in payload:
                continue
            d = payload[model]
            k_list = d["k_list"]
            mid_k = str(k_list[len(k_list) // 2])
            nrc = np.array(d["comp_samples"][mid_k])
            base_probs = np.array(d.get("base_probs", []))
            if len(base_probs) == 0:
                continue   # old-schema cache without base_probs -- nothing to do here
            n = min(len(unc), len(nrc), len(base_probs))
            nrc, bp = nrc[:n], base_probs[:n]
            v = bp * (1 - bp)
            dz = decile_z(nrc, v)

            for col in UNC_COLS:
                if col not in unc.columns:
                    continue
                u = unc[col].values[:n]
                ok = np.isfinite(dz) & np.isfinite(u)
                degenerate = bool(np.ptp(u) < 1e-12) or int(ok.sum()) < 20
                if degenerate:
                    rows.append({
                        "Dataset": ds.upper().replace("_", " "), "Model": model,
                        "Method": method.upper(), "Component": col.replace("_unc", ""),
                        "USFG_decileZ_NRC": np.nan, "CI95_low": np.nan, "CI95_high": np.nan,
                        "CI_excludes_0": False, "MannWhitney_p": np.nan, "rank_biserial": np.nan,
                        "n_Q1": int(ok.sum()), "n_Q4": int(ok.sum()), "n_valid": int(ok.sum()),
                        "Degenerate": True,
                    })
                    continue

                uu, dzz = u[ok], dz[ok]
                q1, q4 = np.percentile(uu, 25), np.percentile(uu, 75)
                low, high = dzz[uu <= q1], dzz[uu >= q4]
                if len(low) < 5 or len(high) < 5:
                    continue

                point, ci_lo, ci_hi = usfg_bootstrap(low, high, dzz)
                stat, p_mw = mannwhitneyu(low, high, alternative="two-sided")
                rank_biserial = 1 - 2 * stat / (len(low) * len(high))

                rows.append({
                    "Dataset": ds.upper().replace("_", " "), "Model": model, "Method": method.upper(),
                    "Component": col.replace("_unc", ""), "USFG_decileZ_NRC": point,
                    "CI95_low": ci_lo, "CI95_high": ci_hi, "CI_excludes_0": bool(ci_lo * ci_hi > 0),
                    "MannWhitney_p": p_mw, "rank_biserial": rank_biserial,
                    "n_Q1": len(low), "n_Q4": len(high), "n_valid": int(ok.sum()),
                    "Degenerate": False,
                })

table9b = pd.DataFrame(rows)
if len(table9b):
    table9b["p_FDR"] = benjamini_hochberg_nan_safe(table9b["MannWhitney_p"].values)
    table9b["Sig_FDR"] = table9b["p_FDR"] < 0.05
    n_fam9b = int(np.isfinite(table9b["MannWhitney_p"].values).sum())
    print(f"FDR family (table9b, decileZ(NRC), MannWhitney_p): {n_fam9b}/{len(table9b)} tests "
          f"actually corrected.")
table9b.to_csv(f"{OUT_DIR}/table9b_usfg_decileZ_nrc.csv", index=False)
print(f"table9b_usfg_decileZ_nrc.csv  ({len(table9b)} rows)")

if len(table9b):
    pivot_b = table9b.pivot_table(index=["Dataset", "Model", "Method"], columns="Component",
                                   values="USFG_decileZ_NRC")
    print("\nUSFG (decileZ-NRC) by component:")
    print(pivot_b.to_string())

if len(table9) and len(table9b):
    merged_cmp = table9.merge(
        table9b, on=["Dataset", "Model", "Method", "Component"], suffixes=("_prob", "_decileZ"))
    if len(merged_cmp):
        print(f"\nOld (raw prob Comp) vs new (decileZ-NRC) USFG, epistemic component only:")
        ce = merged_cmp[merged_cmp["Component"] == "epistemic"]
        print(ce[["Dataset", "Model", "Method", "USFG", "USFG_decileZ_NRC", "Sig_FDR_prob",
                  "Sig_FDR_decileZ"]].to_string(index=False))

FDR family (table9b, decileZ(NRC), MannWhitney_p): 66/96 tests actually corrected.
table9b_usfg_decileZ_nrc.csv  (96 rows)

USFG (decileZ-NRC) by component:
Component                 aleatoric  epistemic     inter     intra
Dataset     Model Method                                          
GMSC        M2    LIME    -0.010274        NaN       NaN       NaN
                  SHAP    -0.024717        NaN       NaN       NaN
            M3    LIME    -0.027636  -0.085309       NaN -0.085309
                  SHAP     0.011362  -0.006465       NaN -0.006465
            M4    LIME    -0.011005  -0.095887 -0.095887       NaN
                  SHAP    -0.028690  -0.008354 -0.008354       NaN
            M5    LIME    -0.038296  -0.112150  0.031059 -0.071321
                  SHAP    -0.032106  -0.008230  0.118759 -0.056789
HOME CREDIT M2    LIME    -0.044269        NaN       NaN       NaN
                  SHAP    -0.031129        NaN       NaN       NaN
            M3    LIME    -0.041752  -0

## 13c. Table 10 — metric orientation diagnostics

v7 audit addition (item 7). Direct diagnostic for FB3 ("the 44.4% issue", CONTEXT §8.3): how far
below `p_null` do the balanced-split base probabilities actually sit, and how does the Random-K
control win rate change as the faithfulness metric is progressively fixed — raw `prob`-variant
Comprehensiveness, then the sign-corrected NRC (`norm` variant), then `decileZ(NRC)`. One row per
(dataset, model, method).


In [33]:
rows = []
for ds in DATASETS:
    for model in UQ_MODELS:
        for method in METHODS:
            payload_prob = load_faithfulness("prob", method, ds)
            payload_norm = load_faithfulness("norm", method, ds)
            if model not in payload_prob or model not in payload_norm:
                continue
            dp, dn = payload_prob[model], payload_norm[model]
            k_list = dp["k_list"]
            mid_k = str(k_list[len(k_list) // 2])

            comp_raw = np.array(dp["comp_samples"][mid_k])
            comp_raw_r = np.array(dp["comp_random_samples"][mid_k])
            winrate_prob_raw = float(np.mean(comp_raw > comp_raw_r))

            base_probs = np.array(dn.get("base_probs", []))
            p_null_samples = np.array(dn.get("p_null_samples", []))
            if len(base_probs) == 0 or len(p_null_samples) == 0:
                rows.append({"Dataset": ds.upper().replace("_", " "), "Model": model,
                             "Method": method.upper(), "K": int(mid_k),
                             "winrate_prob_raw": winrate_prob_raw,
                             "note": "old-schema cache: no base_probs/p_null_samples"})
                continue

            nrc = np.array(dn["comp_samples"][mid_k])
            nrc_r = np.array(dn["comp_random_samples"][mid_k])
            n = min(len(base_probs), len(p_null_samples), len(nrc), len(nrc_r))
            bp, pn, nrc_n, nrc_rn = base_probs[:n], p_null_samples[:n], nrc[:n], nrc_r[:n]

            p_null_mean = float(np.mean(pn))
            pct_below = float(np.mean(bp < pn) * 100)

            # NaN-safe win rate: nrc/nrc_r share the same NaN pattern (both come from the same
            # per-sample null_defined mask), but "nan > nan" silently evaluates to False rather
            # than NaN, so these MUST be masked explicitly before comparing (a plain
            # np.nanmean(nrc > nrc_r) would NOT catch this).
            ok_nrc = np.isfinite(nrc_n) & np.isfinite(nrc_rn)
            if ok_nrc.sum() >= 20:
                winrate_norm_nrc = float(np.mean(nrc_n[ok_nrc] > nrc_rn[ok_nrc]))
                v_ok = bp[ok_nrc] * (1 - bp[ok_nrc])
                dz = decile_z(nrc_n[ok_nrc], v_ok)
                dz_r = decile_z(nrc_rn[ok_nrc], v_ok)
                ok_dz = np.isfinite(dz) & np.isfinite(dz_r)
                winrate_decileZ_nrc = float(np.mean(dz[ok_dz] > dz_r[ok_dz])) if ok_dz.sum() >= 20 else np.nan
            else:
                winrate_norm_nrc = np.nan
                winrate_decileZ_nrc = np.nan

            rows.append({
                "Dataset": ds.upper().replace("_", " "), "Model": model, "Method": method.upper(),
                "K": int(mid_k), "N": n, "p_null_mean": p_null_mean,
                "pct_base_below_pnull": pct_below,
                "winrate_prob_raw": winrate_prob_raw, "winrate_norm_nrc": winrate_norm_nrc,
                "winrate_decileZ_nrc": winrate_decileZ_nrc, "note": "",
            })

table10 = pd.DataFrame(rows)
table10.to_csv(f"{OUT_DIR}/table10_metric_orientation.csv", index=False)
print(f"table10_metric_orientation.csv  ({len(table10)} rows)")
print(table10.to_string(index=False))
print("\nMean Random-K win rate by metric-fix stage (higher = the metric more reliably ranks a "
      "real explanation above a random one; 0.444 was the FB3 headline number for the old, "
      "unfixed metric):")
for col in ["winrate_prob_raw", "winrate_norm_nrc", "winrate_decileZ_nrc"]:
    if col in table10.columns:
        print(f"  {col:22s} mean={table10[col].mean():.4f}")

table10_metric_orientation.csv  (24 rows)
    Dataset Model Method  K    N  p_null_mean  pct_base_below_pnull  winrate_prob_raw  winrate_norm_nrc  winrate_decileZ_nrc note
HOME CREDIT    M2   SHAP 31 2000     0.076755                 37.45            0.6425          0.859029             0.447730     
HOME CREDIT    M2   LIME 31 2000     0.076755                 37.45            0.6430          0.741816             0.468849     
HOME CREDIT    M3   SHAP 31 2000     0.076755                 37.45            0.6425          0.859029             0.447730     
HOME CREDIT    M3   LIME 31 2000     0.076755                 37.45            0.6430          0.741816             0.468849     
HOME CREDIT    M4   SHAP 31 2000     0.079709                 38.70            0.6455          0.890133             0.445333     
HOME CREDIT    M4   LIME 31 2000     0.079709                 38.70            0.6465          0.774400             0.465600     
HOME CREDIT    M5   SHAP 31 2000     0.081448   

## 16. Master summary table

In [34]:
rows = []
for ds in DATASETS:
    ds_label = ds.upper().replace("_", " ")
    for model in UQ_MODELS:
        row = {"Dataset": ds_label, "Model": model}

        perf = table1[(table1["Dataset"] == ds_label) & (table1["Model"] == model) &
                       (table1["Split"] == "balanced")]
        if not perf.empty:
            row["AUC"] = perf["AUC"].values[0]
            row["Epistemic_Pct"] = perf["Epistemic_Pct"].values[0] if "Epistemic_Pct" in perf.columns else np.nan

        g = table4[(table4["Dataset"] == ds_label) & (table4["Model"] == model)]
        if not g.empty:
            row["Gating_AUC_std"] = g["Gating_AUC_standard"].values[0]
            row["AURC"] = g["AURC"].values[0]

        p7 = table7[(table7["Dataset"] == ds_label) & (table7["Model"] == model) &
                     (table7["Variant"] == "prob") & (table7["Method"] == "SHAP")]
        if not p7.empty:
            row["rho_raw_epistemic"] = p7["rho_raw"].values[0]
            row["rho_partial_epistemic"] = p7["rho_partial"].values[0]
            row["rho_aleatoric"] = p7["rho_aleatoric"].values[0]
            row["Sig_after_partial"] = bool(p7["Sig_partial_FDR"].values[0])

        s2b = table2b[(table2b["Dataset"] == ds_label) & (table2b["Model"] == model)]
        if not s2b.empty:
            row["Jaccard_SHAP"] = s2b["Jaccard_SHAP"].values[0]

        rows.append(row)

master = pd.DataFrame(rows)
master.to_csv(f"{OUT_DIR}/table_master_summary.csv", index=False)
print(f"table_master_summary.csv  ({len(master)} rows)")
print(master.to_string(index=False))

print(f"\nAll tables written to {OUT_DIR}")


table_master_summary.csv  (12 rows)
    Dataset Model      AUC  Epistemic_Pct  Gating_AUC_std     AURC  rho_raw_epistemic  rho_partial_epistemic  rho_aleatoric  Sig_after_partial  Jaccard_SHAP
HOME CREDIT    M2 0.725350       0.000000             NaN      NaN                NaN                    NaN       0.766444              False      0.737408
HOME CREDIT    M3 0.726107       1.281644        0.708099 0.058434           0.412546              -0.014832       0.766005              False      0.737408
HOME CREDIT    M4 0.728987       0.619564        0.726288 0.052828           0.486141              -0.010070       0.761719              False      0.743558
HOME CREDIT    M5 0.728529       1.761682        0.719366 0.053994           0.416759              -0.028181       0.759271              False      0.736732
     TAIWAN    M2 0.776195       0.000000             NaN      NaN                NaN                    NaN       0.707840              False      0.825701
     TAIWAN    M3 0.77